Chroma DB Local workflow

Building a traditional Rag system using langchain, chromadb and embedding model (hugging face)

## BUILDING RAG CHAIN USING INBUILT FUNCTIONS

In [3]:
# libraries needed
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

#vectorstore
from langchain_community.vectorstores import Chroma

#other
import numpy as np
from typing import List

C:\Users\Dell\AppData\Local\Temp\ipykernel_11988\1148882588.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Building sample data documents

In [8]:
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine Learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed.
    It focuses on identifying patterns and making predictions based on historical information.
    Machine learning models improve their performance as they are exposed to more data.
    Common types include supervised, unsupervised, and reinforcement learning.
    Applications range from recommendation systems and fraud detection to image recognition and forecasting.
    The quality of data plays a crucial role in model performance.
    Proper evaluation helps ensure reliable and accurate predictions.
    """,

    """
    Deep Learning Fundamentals

    Deep Learning is a specialized subset of machine learning that uses artificial neural networks with multiple layers.
    These networks are inspired by the structure and function of the human brain.
    Deep learning excels at processing large volumes of unstructured data such as images, audio, and text.
    It automatically learns complex features without requiring extensive manual feature engineering.
    Popular architectures include Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs).
    Deep learning has powered advances in computer vision, speech recognition, and generative AI.
    High computational resources are often required for training deep models.
    """,

    """
    Natural Language Processing Fundamentals
    Natural Language Processing (NLP) is a field that enables computers to understand, interpret, and generate human language.
    It combines concepts from linguistics, computer science, and machine learning.
    NLP systems process text and speech to extract meaning and perform useful tasks.
    Common applications include chatbots, language translation, sentiment analysis, and text summarization.
    Modern NLP heavily relies on deep learning models and word embeddings to capture language patterns.
    Transformers have significantly improved the performance of NLP systems.
    Effective NLP solutions require both quality data and appropriate language representations.
    """
]

In [3]:
# saving sample doc to txt files
for i,doc in enumerate(sample_docs):
    with open(f"data/text_files/doc_{i}.txt","w") as f:
        f.write(doc)

 LOADING DOCUMENTS
 

In [9]:
from langchain_community.document_loaders import DirectoryLoader

In [10]:
# loading all text files from the directory
loader = DirectoryLoader(
    path = 'data/text_files',
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)
docs = loader.load()
print(f"No of docs loaded: {len(docs)}")
print(docs)

No of docs loaded: 3
[Document(metadata={'source': 'data\\text_files\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine Learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed.\n    It focuses on identifying patterns and making predictions based on historical information.\n    Machine learning models improve their performance as they are exposed to more data.\n    Common types include supervised, unsupervised, and reinforcement learning.\n    Applications range from recommendation systems and fraud detection to image recognition and forecasting.\n    The quality of data plays a crucial role in model performance.\n    Proper evaluation helps ensure reliable and accurate predictions.\n    '), Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='\n    Deep Learning Fundamentals\n\n    Deep Learning is a specialized subset of machine learning that uses artificial neural n

SPLITTING DOCUMENTS INTO CHUNKS

In [11]:
# initializing splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators = [" "]
)
chunks = splitter.split_documents(docs)
print(f"No of chunks created: {len(chunks)}")


No of chunks created: 6


APPLYING EMBEDDING MODELS and STORING CHUNKS IN CHROMA DB

In [12]:
# initializing embedding model hugging face 
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
# CREATING CHROMA DB VECTOR STORE
persist_dir = "./chroma_db"

# initializing chromadb with hugging face embeddings
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding =  HuggingFaceEmbeddings(),
    persist_directory =  persist_dir,
    collection_name = "rag_collection"
)
print(f"No of vectors created in vector store: {vectorstore._collection.count()}")
print(f"Persisted to: {persist_dir}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

No of vectors created in vector store: 6
Persisted to: ./chroma_db


TESTING SIMILARITY SEARCH

In [9]:
query = " What is NLP?"
similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs


[Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Natural Language Processing Fundamentals\n    Natural Language Processing (NLP) is a field that enables computers to understand, interpret, and generate human language.\n    It combines concepts from linguistics, computer science, and machine learning.\n    NLP systems process text and speech to extract meaning and perform useful tasks.\n    Common applications include chatbots, language translation, sentiment analysis, and text summarization.\n    Modern NLP heavily relies on deep learning'),
 Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Modern NLP heavily relies on deep learning models and word embeddings to capture language patterns.\n    Transformers have significantly improved the performance of NLP systems.\n    Effective NLP solutions require both quality data and appropriate language representations.'),
 Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_cont

ADVANCED SIMILARITY SEARCH WITH SCORES

In [10]:
results_scores = vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Natural Language Processing Fundamentals\n    Natural Language Processing (NLP) is a field that enables computers to understand, interpret, and generate human language.\n    It combines concepts from linguistics, computer science, and machine learning.\n    NLP systems process text and speech to extract meaning and perform useful tasks.\n    Common applications include chatbots, language translation, sentiment analysis, and text summarization.\n    Modern NLP heavily relies on deep learning'),
  0.7757503390312195),
 (Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Modern NLP heavily relies on deep learning models and word embeddings to capture language patterns.\n    Transformers have significantly improved the performance of NLP systems.\n    Effective NLP solutions require both quality data and appropriate language representations.'),
  1.19545316696167),
 (Document(metadata={'sou

INITIALIZING LLM FROM HUGGINGFACE 

In [14]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

INITIALIZING LANGCHAIN CHAT MODEL

In [22]:
from langchain.chat_models.base import init_chat_model

In [13]:
llm = init_chat_model("huggingface:Qwen/Qwen2.5-3B-Instruct")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

RAG CHAIN

In [4]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [15]:
# converting vectorstore to retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001A5F0C3FCB0>, search_kwargs={'k': 3})

In [25]:
# createing  a prompt template
system_prompt = """You are an assitant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you dont know the answer, just say you dont know.
Use three sentences maximum and keep the asnwer concise
Context: {context}"""

prompt =  ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an assitant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you dont know the answer, just say you dont know.\nUse three sentences maximum and keep the asnwer concise\nContext: {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

creating a document chain

In [26]:
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an assitant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you dont know the answer, just say you dont know.\nUse three sentences maximum and keep the asnwer concise\nContext: {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| HuggingFacePipeline(pipeline=TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'cpu', 'input_modalities': 'text

FINAL RAG CHAIN

In [27]:
rag_chain = create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001A719FF7230>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an assitant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf 

In [28]:
rag_chain.invoke({"input":"What is Deep Learning"})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'input': 'What is Deep Learning',
 'context': [Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='Deep Learning Fundamentals\n\n    Deep Learning is a specialized subset of machine learning that uses artificial neural networks with multiple layers.\n    These networks are inspired by the structure and function of the human brain.\n    Deep learning excels at processing large volumes of unstructured data such as images, audio, and text.\n    It automatically learns complex features without requiring extensive manual feature engineering.\n    Popular architectures include Convolutional Neural'),
  Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='architectures include Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs).\n    Deep learning has powered advances in computer vision, speech recognition, and generative AI.\n    High computational resources are often required for training deep models.'),
  Document(metadata

RAG CHAIN USING LCEL (Langchain expression language)

here it is built without inbuilt functions

In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel


In [16]:
# custom prompt for the LLM
custome_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question
If you dont know the answer based on the context, say you dont know.
Provide specific details from the context to support your answer.
Context:
{context}
Question: {question}                                                                                                                                                    
Answer:""")

In [18]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

building the chain using LCEL

In [21]:
rag_chain_lcel = (
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | custome_prompt
    | llm
    | StrOutputParser()
)

In [22]:
response = rag_chain_lcel.invoke("What is Deep Learning")
print(response)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Deep Learning is a specialized subset of machine learning that uses artificial neural networks with multiple layers. The networks are inspired by the structure and function of the human brain and excel at processing large volumes of unstructured data such as images, audio, and text. Deep learning automatically learns complex features without requiring extensive manual feature engineering. Deep learning has powered advances in computer vision, speech recognition, and generative AI. High computational resources are often required for training deep models.

Assistant: Answer: Deep Learning is a specialized subset of machine learning that uses artificial neural networks with multiple layers. The networks are inspired by the structure and function of the human brain and excel at processing large volumes of unstructured data such as images, audio, and text. Deep learning automatically learns complex features without requiring extensive manual feature engineering. Deep learning has powered a

Adding new documents to existing VECTORSTORE

In [23]:
vectorstore

In [28]:
# add temp new document
new_document="""Reinforcement Learning Fundamentals
Reinforcement Learning is a branch of machine learning where an agent learns by interacting with an environment.
The agent takes actions and receives rewards or penalties based on its decisions.
Its objective is to maximize the cumulative reward over time through trial and error.
Key concepts include agents, environments, states, actions, and reward functions.
Unlike supervised learning, reinforcement learning does not rely on labeled datasets for training.
It is widely used in robotics, game playing, autonomous systems, and resource optimization problems.
Popular algorithms include Q-Learning, Deep Q Networks (DQN), and Policy Gradient methods."""

In [29]:
new_document = Document(
    metadata = {
        "source":"manual_addition",
        "topic":"reinforcement_learning"
    },
    page_content=new_document
)
new_document

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning Fundamentals\nReinforcement Learning is a branch of machine learning where an agent learns by interacting with an environment.\nThe agent takes actions and receives rewards or penalties based on its decisions.\nIts objective is to maximize the cumulative reward over time through trial and error.\nKey concepts include agents, environments, states, actions, and reward functions.\nUnlike supervised learning, reinforcement learning does not rely on labeled datasets for training.\nIt is widely used in robotics, game playing, autonomous systems, and resource optimization problems.\nPopular algorithms include Q-Learning, Deep Q Networks (DQN), and Policy Gradient methods.')

In [30]:
# splitting the doc into chunks
new_chunks = splitter.split_documents([new_document])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning Fundamentals\nReinforcement Learning is a branch of machine learning where an agent learns by interacting with an environment.\nThe agent takes actions and receives rewards or penalties based on its decisions.\nIts objective is to maximize the cumulative reward over time through trial and error.\nKey concepts include agents, environments, states, actions, and reward functions.\nUnlike supervised learning, reinforcement learning does not rely on labeled datasets for training.\nIt'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='not rely on labeled datasets for training.\nIt is widely used in robotics, game playing, autonomous systems, and resource optimization problems.\nPopular algorithms include Q-Learning, Deep Q Networks (DQN), and Policy Gradient methods.')]

In [31]:
# adding new documents to vectorstore
vectorstore.add_documents(new_chunks)

['07d244f8-dd65-4c3a-a2c1-5cb82fc2b8cc',
 '0b8de7be-26bb-4c71-9ae2-77fe5495c23c']

In [32]:
# Total count of vectors now
print(f"Total Vectors now: {vectorstore._collection.count()}")

Total Vectors now: 8


Testing by querying updated vectorstore

In [33]:
response = rag_chain_lcel.invoke("What are the key concepts in reinforcement learning ?")
print(response)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 The key concepts in reinforcement learning include agents, environments, states, actions, and reward functions. Based on the provided context, these are the specific details supporting the answer. Reinforcement learning involves an "agent" which interacts with an "environment." Through this interaction, the agent experiences "states," which represent the current situation or condition of the environment. Actions are then taken by the agent based on these states, aiming to maximize the "reward function," which assigns values or points to different outcomes of the agent's actions. These elements form the core framework of reinforcement learning, guiding the agent's decision-making process in order to achieve optimal performance. The context explicitly mentions these five components, further confirming their significance in the field of reinforcement learning. Reinforcement learning does not rely on labeled datasets for training, distinguishing it from other types of machine learning suc